# Tools

Tools are Python functions the model can call. Decorate with `@tool`, give a clear docstring (the model reads it), then pass them into `create_agent`.

**Key learnings**
- Docstrings and type hints become the tool schema the LLM sees
- `@tool("name")` / `description=` customise how the tool is advertised
- Inspect `tool_calls` on AI messages to see what the agent requested


In [ ]:
from dotenv import load_dotenv

load_dotenv()

## Tool definition

Three equivalent ways to declare the same square-root tool: docstring-only, named decorator, and explicit description.


In [ ]:
from langchain.tools import tool

@tool
def square_root(x: float) -> float:
    """Calculate the square root of a number"""
    # Docstring + type hints are converted into the tool schema
    return x ** 0.5


In [ ]:
@tool("square_root")
def tool1(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

In [ ]:
@tool("square_root", description="Calculate the square root of a number")
def tool1(x: float) -> float:
    return x ** 0.5

In [ ]:
tool1.invoke({"x": 467})

## Adding tools to agents

Pass the tool list into `create_agent`. The system prompt should tell the agent *when* to use tools.


In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5-nano",
    tools=[tool1],  # model can call square_root when needed
    system_prompt="You are an arithmetic wizard. Use your tools to calculate the square root and square of any number."
)


In [ ]:
from langchain.messages import HumanMessage

question = HumanMessage(content="What is the square root of 467?")

response = agent.invoke(
    {"messages": [question]}
)

print(response['messages'][-1].content)

### Inspect the tool-calling trace

`pprint` the message list and print `tool_calls` to verify the model chose the right tool and arguments.


In [ ]:
from pprint import pprint

pprint(response['messages'])

In [ ]:
print(response["messages"][1].tool_calls)